# Tier 2: Post-hoc Forgetting Evaluation

This notebook takes the checkpoints saved during Tier 0/1 (full_ft and automode_u10_t10 at seed 8) and evaluates them on MMLU-5shot + ARC-Challenge to measure catastrophic forgetting.

**No retraining** — this is pure eval, typically 1-2 hours per checkpoint.

The claim the paper will make: *AutoMode exhibits less forgetting on general benchmarks than Full-FT, because the LoRA layers constrain most of the weight space.*

Run this **after** Tier 0/1 finish. Can run on any GPU (we put it on GPU 2 since it's often idle after ablations complete).

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'   # pick whichever GPU is free
from pathlib import Path
import json
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

from automode.config import RunConfig
from automode.eval import evaluate_mmlu, evaluate_arc
from automode.train import ensure_dirs, save_json

In [ ]:
# ── Find all saved checkpoints from Tier 0/1 ─────────────────────────────
checkpoint_dirs = []
for runs_root in ['runs/tier0', 'runs/tier1']:
    root = Path(runs_root)
    if not root.exists():
        continue
    for run_dir in root.iterdir():
        ckpt = run_dir / 'checkpoints' / 'final_model'
        cfg_path = run_dir / 'configs' / 'run_config.json'
        complete = run_dir / 'COMPLETE'
        if ckpt.exists() and cfg_path.exists() and complete.exists():
            checkpoint_dirs.append((ckpt, cfg_path))

print(f'Found {len(checkpoint_dirs)} saved checkpoints to evaluate')
for ckpt, cfg_path in checkpoint_dirs[:10]:
    cfg_d = json.load(open(cfg_path))
    print(f'  {cfg_d["model_name"]} | {cfg_d["method"]} | seed={cfg_d["seed"]} | {cfg_d["train_track"]}')

In [ ]:
# ── Eval each checkpoint on MMLU + ARC-C ──────────────────────────────────
import pandas as pd
import gc

CSV = 'runs/tier2_forgetting.csv'
results = []
if Path(CSV).exists():
    done = set(pd.read_csv(CSV)['checkpoint_dir'])
    results = pd.read_csv(CSV).to_dict('records')
else:
    done = set()

for ckpt_dir, cfg_path in checkpoint_dirs:
    if str(ckpt_dir) in done:
        print(f'[skip] {ckpt_dir}')
        continue

    cfg_dict = json.load(open(cfg_path))
    cfg = RunConfig(**{k: v for k, v in cfg_dict.items()
                       if k in RunConfig.__dataclass_fields__})
    cfg.output_root = 'runs/tier2'
    cfg.eval_benchmarks = ('mmlu', 'arc_c')
    cfg.device = 'cuda:0'
    cfg.tier = 2
    # Eval-only configs: trim to a tractable size
    cfg.max_eval_samples = 2000  # MMLU has 14K; 2K is a reasonable subsample
    paths = ensure_dirs(cfg)

    tokenizer = AutoTokenizer.from_pretrained(ckpt_dir)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    print(f'\nLoading {ckpt_dir}...')
    # Detect: did we save a full model (full_ft) or a PEFT adapter (automode/lora)?
    has_adapter = (ckpt_dir / 'adapter_config.json').exists()
    if has_adapter:
        base = AutoModelForCausalLM.from_pretrained(
            cfg.model_name,
            torch_dtype=torch.bfloat16,
            device_map={'': 'cuda:0'},
        )
        model = PeftModel.from_pretrained(base, ckpt_dir)
    else:
        model = AutoModelForCausalLM.from_pretrained(
            ckpt_dir,
            torch_dtype=torch.bfloat16,
            device_map={'': 'cuda:0'},
        )
    model.eval()

    row = {
        'checkpoint_dir': str(ckpt_dir),
        'model_name': cfg.model_name,
        'method': cfg.method,
        'train_track': cfg.train_track,
        'seed': cfg.seed,
    }
    try:
        row.update(evaluate_mmlu(model, tokenizer, cfg, paths))
    except Exception as e:
        print(f'MMLU failed: {e}')
        row['mmlu_5shot_acc'] = None
    try:
        row.update(evaluate_arc(model, tokenizer, cfg, paths))
    except Exception as e:
        print(f'ARC failed: {e}')
        row['arc_c_acc'] = None

    results.append(row)
    pd.DataFrame(results).to_csv(CSV, index=False)
    print(f'{row}')

    del model
    if has_adapter:
        del base
    gc.collect()
    torch.cuda.empty_cache()

print(f'\nDone: {len(results)} checkpoints evaluated')
pd.DataFrame(results)